# ⚡ EasyColab · Google Colab 多模态 AI 推理与生成服务端

本笔记本与 macOS 原生客户端 **`EasyColab.app`** 配套使用，支持一键在 Colab 云端部署多款旗舰开源大模型：
- **Qwen3.8-27B-GGUF**（Unsloth `UD-Q4_K_M` + 原生 `MTP` 多词元投机解码 · 256K 上下文 + 视觉理解）
- **Qwen3.8-27B BF16** / **Qwen-Image-2.1** / **Z-Image-Turbo** / **MiniMax-H3 (33B)**

### ⚡ 运行前准备（重要）：
1. 点击顶部菜单 **`代码执行程序 (Runtime)`** → **`更改运行时类型 (Change runtime type)`**：
   - **硬件加速器**：推荐选择 **`A100 GPU`** 或 **`L4 GPU`**
   - **高内存 (High-RAM)**：强烈建议 **开启**
   - **后台执行 (Background execution)**：建议 **开启**


In [ ]:
#@title 1️⃣ 第一步：检查 Colab GPU 架构与运行环境
import subprocess
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
#@title 2️⃣ 第二步：挂载 Google Drive（持久化模型与生成作品，以后重开 Colab 免重复下载！）
import os
from google.colab import drive

MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/Aevix_Outputs", exist_ok=True)
    os.makedirs("/content/drive/MyDrive/Aevix_Models", exist_ok=True)
    os.environ["DIFFSYNTH_MODEL_BASE_PATH"] = "/content/drive/MyDrive/Aevix_Models"
    print("✅ Google Drive 已挂载！模型将永久保存在云盘 /content/drive/MyDrive/Aevix_Models（并自动兼容读取历史 MiniMax_H3_Models）！")


In [ ]:
#@title 3. 安装 DiffSynth-Studio、最新 Transformers 与多模型高速下载依赖
!pip install -q -U "diffusers>=0.43.0" transformers accelerate bitsandbytes huggingface_hub hf_transfer qwen-vl-utils einops "kernels>=0.17.2,<0.18.0" einops av gradio modelscope imageio imageio-ffmpeg scipy soundfile librosa
!pip install -q --no-deps fla-core flash-linear-attention
!pip install -q --no-deps --only-binary=:all: causal-conv1d || true
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if not os.path.exists("DiffSynth-Studio"):
    !git clone https://github.com/modelscope/DiffSynth-Studio.git
!pip install -q -e "./DiffSynth-Studio"
print("环境与多模型引擎依赖安装完成！")

In [ ]:
%%writefile runtime_auth.py
"""Loopback ingress for the Gradio and Cloudflare tunnels on a Colab VM."""

import contextlib
import os
from pathlib import Path
import re
import shutil
import subprocess
import tempfile
import time
import urllib.request


GRADIO_INGRESS = 7860
GRADIO_BACKEND = 27860
COMFY_INGRESS = 8188
COMFY_BACKEND = 28188
TOYBOX_INGRESS = 8190
TOYBOX_BACKEND = 28190
INFERENCE_PATHS = (
    "/models", "/chat/completions", "/images/generations", "/images/edits", "/videos",
    "/messages/count_tokens", "/messages",
    "/v1/models", "/v1/chat/completions", "/v1/images/generations", "/v1/images/edits", "/v1/videos",
    "/v1/messages/count_tokens", "/v1/messages",
    "/v1/v1/messages/count_tokens", "/v1/v1/messages",
)

# Strict pattern-based inference routes (e.g. /v1/models/{id}, /v1/videos/{id}/content, /v1/images/files/{name}).
# Kept narrow so /v1/models/, /v1/unknown, and traversal paths still fall through to location / (Basic Auth).
INFERENCE_REGEX_PATHS = (
    r"^(?:/v1)?/models/[A-Za-z0-9._-]+(?:/[A-Za-z0-9._-]+)?$",
    r"^(?:/v1)?/videos/[A-Za-z0-9_-]+(?:/content)?$",
    r"^(?:/v1)?/images/files/[A-Za-z0-9._-]+$",
)


def _inference_location(match_expr, backend, suffix="", dotdot_guard=False):
    guard = 'if ($uri ~ "(?:^|/)\\.\\.(?:$|/)") { return 401; }' if dotdot_guard else ""
    return f"""location {match_expr} {{
                auth_basic off;
                {guard}
                if ($request_method = OPTIONS) {{
                    add_header Access-Control-Allow-Origin "*" always;
                    add_header Access-Control-Allow-Methods "GET, POST, DELETE, OPTIONS" always;
                    add_header Access-Control-Allow-Headers "Authorization, Content-Type, X-API-Key, Anthropic-Version, OpenAI-Beta" always;
                    add_header Access-Control-Max-Age 86400 always;
                    return 204;
                }}
                add_header Access-Control-Allow-Origin "*" always;
                add_header Access-Control-Expose-Headers "Content-Disposition" always;
                proxy_set_header Host $http_host;
                proxy_set_header X-Forwarded-Proto https;
                proxy_pass http://127.0.0.1:{backend}{suffix};
            }}"""


def nginx_config(directory):
    # Send canonical inference paths directly to the API gateway (token-authenticated, no Basic Auth
    # popup, CORS enabled for external web platforms). Every other route stays admin-only behind Basic Auth + CSRF checks.
    exact_inference = "\n".join(
        _inference_location(f"= {path}", GRADIO_BACKEND, suffix=path)
        for path in INFERENCE_PATHS
    )
    regex_inference = "\n".join(
        _inference_location(f"~ {pattern}", GRADIO_BACKEND, dotdot_guard=True)
        for pattern in INFERENCE_REGEX_PATHS
    )
    inference = f"{exact_inference}\n{regex_inference}"
    def server(port, backend, extra=""):
        return f"""
        server {{
            listen 127.0.0.1:{port};
            auth_basic "EasyColab";
            auth_basic_user_file "{directory}/users";
            {extra}
            location / {{
                if ($http_sec_fetch_site = cross-site) {{ return 403; }}
                set $origin_check $http_origin;
                if ($http_origin = https://$http_host) {{ set $origin_check ""; }}
                if ($origin_check != "") {{ return 403; }}
                proxy_set_header Host $http_host;
                proxy_set_header X-Forwarded-Proto https;
                proxy_set_header Upgrade $http_upgrade;
                proxy_set_header Connection $connection_upgrade;
                proxy_set_header Authorization "";
                proxy_pass http://127.0.0.1:{backend};
            }}
        }}
        """
    return f"""
    daemon off;
    master_process off;
    pid "{directory}/nginx.pid";
    error_log stderr warn;
    events {{ worker_connections 1024; }}
    http {{
        access_log off;
        map $http_upgrade $connection_upgrade {{ default upgrade; '' ''; }}
        client_max_body_size 0;
        proxy_http_version 1.1;
        proxy_buffering off;
        proxy_request_buffering off;
        proxy_read_timeout 3600s;
        proxy_hide_header Access-Control-Allow-Origin;
        {server(GRADIO_INGRESS, GRADIO_BACKEND, inference)}
        {server(COMFY_INGRESS, COMFY_BACKEND)}
        {server(TOYBOX_INGRESS, TOYBOX_BACKEND)}
    }}
    """


@contextlib.contextmanager
def authenticated_ingress(password):
    if not re.fullmatch(r"[A-Za-z0-9_-]{24,256}", password):
        raise ValueError("Management password must contain at least 24 URL-safe ASCII characters")
    nginx = shutil.which("nginx")
    if not nginx:
        raise RuntimeError("Install nginx on the runtime before starting EasyColab")
    with tempfile.TemporaryDirectory(prefix="easycolab-auth-") as directory:
        digest = subprocess.run(
            ["openssl", "passwd", "-6", "-stdin"], input=password, text=True,
            capture_output=True, check=True,
        ).stdout.strip()
        Path(directory, "users").write_text(f"easycolab:{digest}\n", encoding="ascii")
        config = Path(directory, "nginx.conf")
        config.write_text(nginx_config(directory), encoding="utf-8")
        with subprocess.Popen([nginx, "-p", directory, "-c", str(config)]) as proxy:
            try:
                # A wrong password must be challenged by both listeners before
                # either tunnel is published. Fail if a previous proxy owns a port.
                opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
                for port in (GRADIO_INGRESS, COMFY_INGRESS, TOYBOX_INGRESS):
                    for _ in range(100):
                        if proxy.poll() is not None:
                            raise RuntimeError("Authenticated ingress failed to bind")
                        try:
                            opener.open(f"http://127.0.0.1:{port}/").close()
                        except urllib.error.HTTPError as error:
                            if error.code == 401:
                                break
                            raise
                        except urllib.error.URLError:
                            time.sleep(0.05)
                            continue
                        raise RuntimeError("Ingress accepted anonymous traffic")
                    else:
                        raise RuntimeError("Authenticated ingress did not start")
                # nginx retries occupied ports for a while before exiting. Its
                # private pid file appears only after both sockets are bound.
                if not Path(directory, "nginx.pid").exists():
                    raise RuntimeError("Ingress ports are already occupied")
                yield
            finally:
                proxy.terminate()


In [ ]:
#@title 4️⃣ 第四步：写入 EasyColab 模块化服务端包 (easycolab_server/*) 与主程序 (app.py)
import base64, json, os, shutil, zlib

_BUNDLE_B64 = "..."
_pkg_files = json.loads(zlib.decompress(base64.b64decode(_BUNDLE_B64)).decode("utf-8"))
shutil.rmtree("easycolab_server", ignore_errors=True)
os.makedirs("easycolab_server", exist_ok=True)
for _name, _code in _pkg_files.items():
    with open(os.path.join("easycolab_server", os.path.basename(_name)), "w", encoding="utf-8") as _f:
        _f.write(_code)
with open("app.py", "w", encoding="utf-8") as _af:
    _af.write("\"\"\"EasyColab 服务端主入口与向后兼容模块门面。\n\n所有核心业务逻辑已按职责拆分至 `easycolab_server` 子模块包中；\n本文件保留 CLI 启动编排（`--local` 本地预览与 Colab 鉴权隧道部署）及对历史单文件导出符号的兼容代理。\n\"\"\"\n\nimport os\nimport sys\nimport contextlib\nimport signal\nimport types\nimport urllib.parse\n\n# Remove the deployment secret before any submodule or model installer starts subprocesses.\n_DEPLOY_PASSWORD = os.environ.pop(\"EASYCOLAB_ADMIN_PASSWORD\", \"\") if __name__ == \"__main__\" else \"\"\n\nfrom easycolab_server import *  # noqa: F401,F403\nfrom easycolab_server import runtime_state as _rs\n\n# 需与 easycolab_server.runtime_state 保持实时双向同步的可变全局状态名称\n_SYNCED_STATE_ATTRS = frozenset(\n    {\n        \"SELECTED_MODEL_ID\",\n        \"LOADED_MODEL_ID\",\n        \"CURRENT_MODEL_MODE\",\n        \"PIPELINE\",\n        \"LLM_BUNDLE\",\n        \"QWEN_CONTEXT_LENGTH\",\n        \"API_AUTH_KEY\",\n        \"EXTRA_API_TOKENS\",\n        \"API_REQUEST_COUNT\",\n        \"IDLE_TIMEOUT_MINUTES\",\n        \"LAST_ACTIVE_TIME\",\n        \"_ACTIVE_API_STREAMS\",\n        \"CURRENT_STAGE_BAR_LABEL\",\n        \"GRADIO_BACKEND\",\n        \"GRADIO_INGRESS\",\n        \"COMFY_BACKEND\",\n        \"COMFY_INGRESS\",\n        \"TOYBOX_BACKEND\",\n        \"TOYBOX_INGRESS\",\n        \"DEFAULT_OUTPUT_DIR\",\n        \"_DEPLOY_PASSWORD\",\n    }\n)\n\n\nclass _AppCompatModule(types.ModuleType):\n    \"\"\"将可变状态标量读写实时代理至 easycolab_server.runtime_state，保障单文件历史引用与测试兼容。\"\"\"\n\n    def __getattribute__(self, name: str):\n        if name in _SYNCED_STATE_ATTRS:\n            return getattr(_rs, name)\n        return super().__getattribute__(name)\n\n    def __setattr__(self, name: str, value):\n        if name in _SYNCED_STATE_ATTRS:\n            setattr(_rs, name, value)\n        super().__setattr__(name, value)\n\n\nsys.modules[__name__].__class__ = _AppCompatModule\n\n\nif __name__ == \"__main__\":\n    import getpass\n    from gradio import networking\n\n    def stop_server(signum, frame):\n        raise SystemExit(0)\n\n    signal.signal(signal.SIGTERM, stop_server)\n\n    if _IS_LOCAL_PREVIEW:\n        local_port = int(os.environ.get(\"EASYCOLAB_PORT\", str(_rs.GRADIO_INGRESS)))\n        ui = create_ui()\n        ui.queue()\n        app, _, _ = ui.launch(\n            server_name=\"127.0.0.1\",\n            server_port=local_port,\n            share=False,\n            inbrowser=True,\n            inline=False,\n            prevent_thread_lock=True,\n            css=CUSTOM_CSS,\n        )\n        attach_macos_api_routes(app)\n        print(f\"🌐 EasyColab 本地网页调试端已启动: http://127.0.0.1:{local_port}\", flush=True)\n        try:\n            ui.block_thread()\n        finally:\n            ui.close()\n    else:\n        if authenticated_ingress is not None and (_DEPLOY_PASSWORD or sys.stdin.isatty()):\n            password = _DEPLOY_PASSWORD or getpass.getpass(\"Management password (24+ URL-safe characters): \")\n            ingress_ctx = authenticated_ingress(password)\n        else:\n            ingress_ctx = contextlib.nullcontext()\n\n        with ingress_ctx:\n            ui = create_ui()\n            ui.queue()\n            app, _, _ = ui.launch(\n                server_name=\"127.0.0.1\",\n                server_port=_rs.GRADIO_BACKEND,\n                share=False,\n                inline=False,\n                prevent_thread_lock=True,\n                css=CUSTOM_CSS,\n            )\n            attach_macos_api_routes(app)\n            tunnel = networking.setup_tunnel(\n                local_host=\"127.0.0.1\",\n                local_port=_rs.GRADIO_INGRESS,\n                share_token=ui.share_token,\n                share_server_address=None,\n                share_server_tls_certificate=None,\n            )\n            share_url = urllib.parse.urlsplit(tunnel)._replace(scheme=\"https\").geturl()\n            print(f\"EasyColab URL: {share_url}\", flush=True)\n            _runtime_meta_file = \"/content/.easycolab_runtime.json\"\n            _managed_marker_file = \"/content/.easycolab_managed.json\"\n            if os.path.isdir(\"/content\"):\n                try:\n                    import hashlib as _hashlib\n                    import json as _json\n                    with open(_managed_marker_file, \"w\", encoding=\"utf-8\") as _mf:\n                        _json.dump({\n                            \"managed_by\": \"easycolab\",\n                            \"instance_name\": _rs.INSTANCE_NAME or \"easycolab\",\n                            \"updated_at\": time.time(),\n                        }, _mf, ensure_ascii=False)\n                    _pw_digest = _hashlib.sha256(_DEPLOY_PASSWORD.encode(\"utf-8\")).hexdigest() if _DEPLOY_PASSWORD else \"\"\n                    _tmp_rt = _runtime_meta_file + \".tmp\"\n                    with open(_tmp_rt, \"w\", encoding=\"utf-8\") as _rf:\n                        _json.dump({\n                            \"managed_by\": \"easycolab\",\n                            \"pid\": os.getpid(),\n                            \"instance_name\": _rs.INSTANCE_NAME or \"easycolab\",\n                            \"bundle_hash\": _rs.BUNDLE_HASH,\n                            \"share_url\": share_url,\n                            \"password_sha256\": _pw_digest,\n                            \"started_at\": time.time(),\n                        }, _rf, ensure_ascii=False)\n                    os.replace(_tmp_rt, _runtime_meta_file)\n                except Exception:\n                    pass\n            try:\n                ui.block_thread()\n            finally:\n                if os.path.isfile(_runtime_meta_file):\n                    try:\n                        import json as _json\n                        with open(_runtime_meta_file, \"r\", encoding=\"utf-8\") as _rf:\n                            _cur_meta = _json.load(_rf)\n                        if int(_cur_meta.get(\"pid\") or -1) == os.getpid():\n                            os.remove(_runtime_meta_file)\n                    except Exception:\n                        pass\n                ui.close()\n")
print(f"✅ 已写入 easycolab_server ({len(_pkg_files)} 个模块) 与 app.py")


In [ ]:
#@title Start the authenticated EasyColab service
import os, getpass, subprocess, sys, shutil
if not shutil.which("nginx"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "nginx-light"], check=True)
    subprocess.run(["service", "nginx", "stop"], check=False)
subprocess.run([sys.executable, "app.py"], env={**os.environ, "EASYCOLAB_ADMIN_PASSWORD": getpass.getpass("Management password (24+ URL-safe characters): ")}, check=True)


In [ ]:
#@title 5️⃣ 第五步：用完后一键释放 Colab 虚拟机（省 Google AI Ultra 计算单元）
from google.colab import runtime
print("🛑 正在断开并释放 Colab 运行时，停止扣除计算单元...")
runtime.unassign()